# C 모델 단독 — dashcam_anonymizer (YOLOv8) 얼굴·번호판 모자이크

비교 노트북의 **C 항목만** 떼어낸 실행용 버전입니다.

| 항목 | 내용 |
|---|---|
| 모델 | `dashcam_anonymizer` 의 `best.pt` (YOLOv8, 얼굴+번호판 **단일 모델**) |
| 가중치 | Google Drive ID `1uV8IMuGDbmDabdjyeSy4SUKV9OS-ULbe` (gdown 자동 수령) |
| 입력 | **이미지 · 영상 · 폴더(일괄)** 전부 지원 |
| 출력 | 원본과 같은 형식으로 `_mosaic` 접미사를 붙여 저장 |

**이미지도 됩니다.** 모델은 프레임 1장 단위로 추론하므로 영상이든 PNG/JPG든 동작이 같습니다.
`INPUT_PATH`에 이미지 경로를 넣으면 이미지 1장을, 폴더를 넣으면 그 안의 이미지·영상을 전부 처리합니다.

원본 대비 추가한 것: **폴더 일괄 처리**, **클래스별 conf·확대비율 분리**, **face/plate 개별 토글**,
**프레임 스킵(DETECT_EVERY)**, **처리 건별 요약 표**.


---
## 0. 설정 — 여기만 수정하세요

In [1]:
from pathlib import Path
import os, sys, time, json, shutil, subprocess

# ===== 여기만 수정 =====
# 파일(이미지/영상) 또는 폴더 아무거나 OK.
INPUT_PATH  = Path(r"C:\Users\Win11Pro\Desktop\KakaoTalk_20260731_110750941.png")   # 이미지 / 영상 / 폴더
OUTPUT_DIR  = Path(r"C:\Users\Win11Pro\Desktop\새 폴더")     # 결과 저장 폴더
WORK_DIR    = Path(r"C:\Users\Win11Pro\Desktop\새 폴더")     # 가중치 다운로드 폴더

DEVICE      = "0"          # GPU면 "0", CPU면 "cpu"
MOSAIC_MODE = "pixelate"   # "pixelate"(모자이크) | "gaussian"(블러)
BLOCKS      = 12           # 모자이크 격자 수. 클수록 잘게(약하게) 가려짐
IMGSZ       = 1280         # 추론 해상도. 작은 얼굴을 놓치면 1600까지 올려보세요

# 무엇을 가릴지 (모델 클래스명에서 자동 매칭)
BLUR_FACE   = True
BLUR_PLATE  = True

# 클래스별 임계값 / 박스 확대비율 (비식별화는 recall 우선 → conf를 낮게)
CONF_FACE   = 0.10
CONF_PLATE  = 0.10
SCALE_FACE  = 1.25         # 머리카락·턱선까지 덮이도록 얼굴은 조금 더 크게
SCALE_PLATE = 1.15

# 영상 전용 옵션
MAX_FRAMES   = 300         # 먼저 짧게 테스트. 전체 처리하려면 None
DETECT_EVERY = 1           # 1=매 프레임 탐지. 2~3이면 그만큼 빨라지되 빠른 움직임에서 놓칠 수 있음

# 폴더 입력일 때
RECURSIVE   = False        # True면 하위 폴더까지 훑음
OVERWRITE   = False        # False면 이미 만들어진 결과는 건너뜀
# ======================

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
WORK_DIR.mkdir(parents=True, exist_ok=True)
MODELS_DIR = WORK_DIR / "models"; MODELS_DIR.mkdir(exist_ok=True)

IMG_EXT = {".png", ".jpg", ".jpeg", ".bmp", ".webp", ".tif", ".tiff"}
VID_EXT = {".mp4", ".avi", ".mov", ".mkv", ".wmv", ".flv"}

if not INPUT_PATH.exists():
    raise FileNotFoundError(f"입력 경로가 없습니다: {INPUT_PATH}")

def kind_of(p: Path):
    e = p.suffix.lower()
    return "image" if e in IMG_EXT else "video" if e in VID_EXT else None

if INPUT_PATH.is_dir():
    it = INPUT_PATH.rglob("*") if RECURSIVE else INPUT_PATH.glob("*")
    TARGETS = sorted(p for p in it if p.is_file() and kind_of(p))
    if not TARGETS:
        raise FileNotFoundError(f"폴더에 처리할 이미지/영상이 없습니다: {INPUT_PATH}")
else:
    if kind_of(INPUT_PATH) is None:
        raise ValueError(f"지원하지 않는 확장자: {INPUT_PATH.suffix}")
    TARGETS = [INPUT_PATH]

print(f"처리 대상 {len(TARGETS)}건")
for p in TARGETS[:10]:
    print(f"  - [{kind_of(p)}] {p.name}")
if len(TARGETS) > 10:
    print(f"  ... 외 {len(TARGETS)-10}건")
print("출력:", OUTPUT_DIR)

처리 대상 1건
  - [image] KakaoTalk_20260731_110750941.png
출력: C:\Users\Win11Pro\Desktop\새 폴더


---
## 1. 설치 & 환경 확인

In [2]:
%pip install -q ultralytics opencv-python gdown pandas matplotlib

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.0 -> 26.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [3]:
import cv2, torch, numpy as np
from ultralytics import YOLO

print("torch", torch.__version__, "| cuda:", torch.cuda.is_available(),
      "|", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU only")
print("opencv", cv2.__version__)

# GPU가 없는데 DEVICE="0"이면 조용히 실패하지 않도록 여기서 바로잡습니다.
if DEVICE != "cpu" and not torch.cuda.is_available():
    print("[warn] CUDA 사용 불가 → DEVICE를 'cpu'로 전환합니다.")
    DEVICE = "cpu"

torch 2.12.0.dev20260408+cu128 | cuda: True | NVIDIA GeForce RTX 5060 Laptop GPU
opencv 4.10.0


---
## 2. 유틸 — 한글 경로 안전 입출력 + 모자이크

In [ ]:
def imread_u(path):
    """한글/공백 경로에서도 안전하게 읽기 (cv2.imread는 유니코드 경로에 약함)."""
    frame = cv2.imdecode(np.fromfile(str(path), dtype=np.uint8), cv2.IMREAD_COLOR)
    if frame is None:
        raise RuntimeError(f"이미지를 읽을 수 없습니다: {path}")
    return frame

def imwrite_u(path, frame):
    ok, buf = cv2.imencode(Path(path).suffix, frame)
    if not ok:
        raise RuntimeError(f"이미지 인코딩 실패: {path}")
    buf.tofile(str(path))

def pixelate(img, x1, y1, x2, y2, blocks=BLOCKS):
    h, w = img.shape[:2]
    x1, y1 = max(0, int(x1)), max(0, int(y1))
    x2, y2 = min(w, int(x2)), min(h, int(y2))
    if x2 <= x1 or y2 <= y1:
        return img
    roi = img[y1:y2, x1:x2]
    small = cv2.resize(roi, (max(1, (x2-x1)//blocks), max(1, (y2-y1)//blocks)),
                       interpolation=cv2.INTER_LINEAR)
    img[y1:y2, x1:x2] = cv2.resize(small, (x2-x1, y2-y1), interpolation=cv2.INTER_NEAREST)
    return img

def gaussian(img, x1, y1, x2, y2):
    h, w = img.shape[:2]
    x1, y1 = max(0, int(x1)), max(0, int(y1))
    x2, y2 = min(w, int(x2)), min(h, int(y2))
    if x2 <= x1 or y2 <= y1:
        return img
    k = max(3, (min(x2-x1, y2-y1)//3) | 1)   # 커널은 반드시 홀수
    img[y1:y2, x1:x2] = cv2.GaussianBlur(img[y1:y2, x1:x2], (k, k), 0)
    return img

APPLY = pixelate if MOSAIC_MODE == "pixelate" else gaussian

def expand(box, scale):
    """중심 고정으로 박스를 scale배 확대. 경계에 얼굴 일부가 남는 걸 막습니다."""
    x1, y1, x2, y2 = box
    cx, cy = (x1+x2)/2, (y1+y2)/2
    bw, bh = (x2-x1)*scale, (y2-y1)*scale
    return cx-bw/2, cy-bh/2, cx+bw/2, cy+bh/2

def video_meta(path: Path):
    cap = cv2.VideoCapture(str(path))
    if not cap.isOpened():
        raise RuntimeError(f"영상을 열 수 없습니다: {path}")
    m = dict(fps=cap.get(cv2.CAP_PROP_FPS) or 30.0,
             w=int(cap.get(cv2.CAP_PROP_FRAME_WIDTH)),
             h=int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT)),
             n=int(cap.get(cv2.CAP_PROP_FRAME_COUNT)))
    cap.release()
    return m

print("유틸 준비 완료 ·", "모자이크 방식:", MOSAIC_MODE)

---
## 3. 가중치 수령 & 모델 로드

`gdown`이 할당량 초과로 막히면(하루 다운로드 제한) 셀이 안내하는 링크로 브라우저에서 직접 받아
`models/dashcam_best.pt` 로 저장한 뒤 이 셀만 다시 실행하시면 됩니다.

In [ ]:
DC_W = MODELS_DIR / "dashcam_best.pt"
GDRIVE_ID = "1uV8IMuGDbmDabdjyeSy4SUKV9OS-ULbe"   # dashcam_anonymizer setup.sh의 실제 ID

def run(cmd, cwd=None):
    print(">", " ".join(map(str, cmd)))
    p = subprocess.run([str(c) for c in cmd], cwd=cwd, capture_output=True, text=True)
    if p.stdout: print(p.stdout[-2000:])
    if p.returncode != 0: print("[stderr]", p.stderr[-2000:])
    return p.returncode

if not (DC_W.exists() and DC_W.stat().st_size > 5e6):
    run([sys.executable, "-m", "gdown", GDRIVE_ID, "-O", str(DC_W)])

if not (DC_W.exists() and DC_W.stat().st_size > 5e6):
    DC_W.unlink(missing_ok=True)   # 404 HTML이 .pt로 남는 사고 방지
    raise FileNotFoundError(
        "가중치 수령 실패. 브라우저로 직접 받으세요:\n"
        f"  https://drive.google.com/uc?id={GDRIVE_ID}\n"
        f"  받은 best.pt 를 → {DC_W} 로 이름 바꿔 저장 후 이 셀 재실행")

print(f"[ok] {DC_W.name} ({DC_W.stat().st_size/1e6:.1f} MB)")
model = YOLO(str(DC_W))
print("classes:", model.names)

In [ ]:
# 클래스명 → (사용여부, conf, scale) 매핑을 자동 생성.
# 이 모델은 얼굴/번호판 클래스명이 레포마다 조금씩 다를 수 있어 이름으로 판별합니다.
FACE_KEYS  = ("face", "head", "person_face")
PLATE_KEYS = ("plate", "lp", "license", "number")

CLASS_CFG = {}
for idx, name in model.names.items():
    low = str(name).lower()
    if any(k in low for k in FACE_KEYS):
        CLASS_CFG[idx] = dict(group="face",  use=BLUR_FACE,  conf=CONF_FACE,  scale=SCALE_FACE)
    elif any(k in low for k in PLATE_KEYS):
        CLASS_CFG[idx] = dict(group="plate", use=BLUR_PLATE, conf=CONF_PLATE, scale=SCALE_PLATE)
    else:   # 정체불명 클래스는 안전하게 얼굴 기준으로 처리
        CLASS_CFG[idx] = dict(group="other", use=True, conf=min(CONF_FACE, CONF_PLATE), scale=SCALE_FACE)

ACTIVE = [i for i, c in CLASS_CFG.items() if c["use"]]
BASE_CONF = min(c["conf"] for c in CLASS_CFG.values())   # 낮게 뽑고 클래스별로 다시 거릅니다

for i, c in CLASS_CFG.items():
    print(f"  [{i}] {model.names[i]:<16} group={c['group']:<6} use={c['use']} conf={c['conf']} scale={c['scale']}")
if not ACTIVE:
    raise ValueError("BLUR_FACE / BLUR_PLATE 가 모두 False입니다. 가릴 대상이 없습니다.")

---
## 4. 처리 함수 — 이미지 / 영상 공통

In [ ]:
def detect_boxes(frame):
    """반환: [(x1,y1,x2,y2, scale, group), ...] — 클래스별 conf로 필터링된 결과"""
    r = model.predict(frame, conf=BASE_CONF, imgsz=IMGSZ, device=DEVICE,
                      classes=ACTIVE, verbose=False)[0]
    out = []
    if r.boxes is None or len(r.boxes) == 0:
        return out
    xyxy = r.boxes.xyxy.cpu().numpy()
    cls  = r.boxes.cls.cpu().numpy().astype(int)
    conf = r.boxes.conf.cpu().numpy()
    for (x1, y1, x2, y2), c, s in zip(xyxy, cls, conf):
        cfg = CLASS_CFG.get(int(c))
        if cfg and cfg["use"] and s >= cfg["conf"]:
            out.append((x1, y1, x2, y2, cfg["scale"], cfg["group"]))
    return out

def blur_frame(frame, boxes):
    for x1, y1, x2, y2, scale, _ in boxes:
        frame = APPLY(frame, *expand((x1, y1, x2, y2), scale))
    return frame

def out_path_for(src: Path) -> Path:
    """원본 확장자를 유지하되 영상은 mp4v 코덱이라 .mp4로 통일."""
    if kind_of(src) == "image":
        return OUTPUT_DIR / f"{src.stem}_mosaic{src.suffix}"
    return OUTPUT_DIR / f"{src.stem}_mosaic.mp4"

def process_image(src: Path, dst: Path):
    frame = imread_u(src)
    boxes = detect_boxes(frame)
    imwrite_u(dst, blur_frame(frame, boxes))
    counts = {}
    for b in boxes:
        counts[b[5]] = counts.get(b[5], 0) + 1
    return dict(frames=1, hits=len(boxes), detail=counts)

def process_video(src: Path, dst: Path):
    meta = video_meta(src)
    cap = cv2.VideoCapture(str(src))
    vw = cv2.VideoWriter(str(dst), cv2.VideoWriter_fourcc(*"mp4v"),
                         meta["fps"], (meta["w"], meta["h"]))
    if not vw.isOpened():
        cap.release()
        raise RuntimeError("VideoWriter 열기 실패 (코덱 문제). 셀 하단 트러블슈팅 표 참고")
    n = hits = 0
    counts, cached = {}, []
    try:
        while True:
            ok, frame = cap.read()
            if not ok or (MAX_FRAMES and n >= MAX_FRAMES):
                break
            if n % max(1, DETECT_EVERY) == 0:
                cached = detect_boxes(frame)     # 탐지 프레임
            for b in cached:
                counts[b[5]] = counts.get(b[5], 0) + 1
            frame = blur_frame(frame, cached)
            hits += len(cached)
            vw.write(frame); n += 1
            if n % 100 == 0:
                print(f"    {n} frames / 누적 검출 {hits}")
    finally:
        cap.release(); vw.release()
    return dict(frames=n, hits=hits, detail=counts)

print("처리 함수 준비 완료")

---
## 5. 실행

In [ ]:
RESULTS = []
t_all = time.time()

for i, src in enumerate(TARGETS, 1):
    dst = out_path_for(src)
    t0 = time.time()
    print(f"[{i}/{len(TARGETS)}] {src.name} → {dst.name}")

    if dst.exists() and not OVERWRITE:
        print("    이미 결과가 있어 건너뜁니다 (OVERWRITE=True로 덮어쓰기)")
        continue

    try:
        info = process_image(src, dst) if kind_of(src) == "image" else process_video(src, dst)
        RESULTS.append(dict(입력=src.name, 종류=kind_of(src), 성공=True,
                            프레임=info["frames"], 검출=info["hits"],
                            상세=", ".join(f"{k}:{v}" for k, v in info["detail"].items()) or "-",
                            초=round(time.time()-t0, 1), 출력=str(dst)))
        print(f"    OK · {info['frames']}프레임 / 검출 {info['hits']} · {round(time.time()-t0,1)}s")
    except Exception as e:
        RESULTS.append(dict(입력=src.name, 종류=kind_of(src), 성공=False,
                            프레임=0, 검출=0, 상세=f"{type(e).__name__}: {e}",
                            초=round(time.time()-t0, 1), 출력=""))
        print(f"    FAIL · {type(e).__name__}: {e}")

print(f"\n전체 완료 · {round(time.time()-t_all, 1)}s")

In [ ]:
import pandas as pd
df = pd.DataFrame(RESULTS)
display(df if len(df) else "처리된 항목이 없습니다 (전부 건너뜀?)")

(OUTPUT_DIR / "summary_C.json").write_text(
    json.dumps(RESULTS, ensure_ascii=False, indent=2), encoding="utf-8")
print("저장:", OUTPUT_DIR / "summary_C.json")

# 검출 0건이면 대부분 conf/해상도 문제입니다.
if len(df) and df["검출"].sum() == 0:
    print("\n[점검] 검출 0건 → CONF_FACE/CONF_PLATE를 0.05로, IMGSZ를 1600으로 올려 재실행해 보세요.")

---
## 6. 원본 vs 결과 미리보기

In [ ]:
import matplotlib.pyplot as plt

PREVIEW_IDX  = 0    # RESULTS 중 몇 번째 항목을 볼지
FRAME_IDX    = 50   # 영상일 때 확인할 프레임 번호 (MAX_FRAMES보다 작게)

def grab(path, idx):
    p = Path(path)
    if p.suffix.lower() in IMG_EXT:
        return imread_u(p)
    cap = cv2.VideoCapture(str(p))
    cap.set(cv2.CAP_PROP_POS_FRAMES, idx)
    ok, f = cap.read(); cap.release()
    return f if ok else None

ok_rows = [r for r in RESULTS if r["성공"]]
if not ok_rows:
    print("성공한 결과가 없어 미리보기를 건너뜁니다.")
else:
    row = ok_rows[min(PREVIEW_IDX, len(ok_rows)-1)]
    src = next(p for p in TARGETS if p.name == row["입력"])
    pair = [("원본", src), ("모자이크", Path(row["출력"]))]

    fig, axes = plt.subplots(1, 2, figsize=(14, 6))
    for ax, (title, p) in zip(axes, pair):
        img = grab(p, FRAME_IDX)
        if img is None:
            ax.text(.5, .5, "read fail", ha="center")
        else:
            ax.imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
        ax.set_title(f"{title} — {p.name}", fontsize=11); ax.axis("off")
    plt.tight_layout(); plt.show()

---
## 7. 잘 안 될 때

| 증상 | 조치 |
|---|---|
| gdown 실패 (할당량 초과) | `https://drive.google.com/uc?id=1uV8IMuGDbmDabdjyeSy4SUKV9OS-ULbe` 로 직접 받아 `models/dashcam_best.pt` 로 저장 |
| 검출 0건 | `CONF_FACE=CONF_PLATE=0.05`, `IMGSZ=1600` |
| 작은 얼굴을 놓침 | `IMGSZ=1600~1920`. 4K 원본이면 특히 효과가 큽니다 |
| 경계에 얼굴 일부가 남음 | `SCALE_FACE=1.35` |
| 모자이크가 너무 약함 | `BLOCKS=6` (숫자가 작을수록 굵게 뭉개짐) |
| CUDA out of memory | `IMGSZ`를 960 또는 640으로 |
| 결과 mp4가 0바이트 | 코덱 문제. `VideoWriter_fourcc(*"mp4v")` → `*"XVID"` + 출력 확장자 `.avi` |
| 결과 영상에 소리가 없음 | OpenCV는 오디오를 못 씁니다. 필요하면 `ffmpeg -i out.mp4 -i src.mp4 -c copy -map 0:v -map 1:a final.mp4` |
| 처리가 느림 | `DETECT_EVERY=2~3`, `IMGSZ=960`. 정지 화면 위주면 손실이 거의 없습니다 |

### 다음 단계로 좋은 것

- **국내 도메인 보강**: 이 모델은 해외 대시캠 데이터 기반이라 한국 번호판(초록 구형/흰색 신형)에서 recall이 떨어질 수 있습니다. 갖고 계신 AI-Hub 차량 데이터에 번호판 bbox를 라벨링해 `best.pt`를 시작 가중치로 파인튜닝하면 바로 개선됩니다.
- **트래킹 결합**: ByteTrack을 붙여 프레임 간 박스를 이어주면 한두 프레임 놓친 구간도 메워집니다. 비식별화는 precision보다 **recall**이 전부라 이게 실효가 큽니다.
- **실시간 적용**: 이미 만들어 둔 `realtime_anonymizer.py`가 같은 모델을 쓰므로, 여기서 정한 `CONF`/`SCALE` 값을 그대로 옮기면 됩니다.
